# Cost-Efficiency Simulator

**Scenario:** Your team is evaluating Latence for production. The question isn't whether it works — it's whether it's cost-effective at scale. You need to model costs for realistic workloads, compare with/without optional features (refinement, TOON, entity resolution), and project monthly budgets.

**What we'll build:**
1. Latence pricing reference (all services, base + addon costs)
2. Simulate a 10K-document pipeline: DocInt → Entity Extraction → Relation Extraction
3. Compression ROI: what you spend on compression vs what you save on LLM tokens
4. Feature cost comparison: base vs with refinement/optimization addons
5. Monthly budget projections for different workload sizes

**Services used:** Credits API (for live balance check), pricing math

**Prerequisites:**
- `pip install latence`
- `LATENCE_API_KEY` environment variable set

In [ ]:
import sys
sys.path.insert(0, "..")
from _test_utils import setup_notebook, print_section, print_subsection, print_success

client = setup_notebook()

# Check current balance
balance = client.credits.balance()
print(f"Current balance: ${balance.balance_usd:.2f}")

## Latence Pricing Reference

All pricing is per-unit (tokens or pages), pay-as-you-go. No minimums, no commitments.

In [ ]:
# Latence pricing as of Feb 2025 (USD)
PRICING = {
    "embedding": {
        "unit": "1M tokens",
        "base": 0.10,
    },
    "colbert": {
        "unit": "1M tokens",
        "query": 0.10,
        "document": 0.40,
    },
    "colpali": {
        "unit": "varies",
        "query": 0.10,  # per 1M tokens
        "image": 1.00,  # per 1K images
    },
    "document_intelligence": {
        "unit": "1K pages",
        "default": 5.00,
        "performance": 7.50,
        "addons": {"layout_detection": 1.25},
    },
    "extraction": {
        "unit": "1M tokens",
        "base": 1.00,
        "addons": {"auto_label": 5.00, "label_refinement": 10.00},
    },
    "redaction": {
        "unit": "1M tokens",
        "base": 1.50,
        "addons": {"synthetic_pii": 5.00, "llm_refinement": 10.00},
    },
    "compression": {
        "unit": "1M tokens",
        "base": 0.25,
        "addons": {"chat_annealing": 0.25, "toon": 0.50},
    },
    "ontology": {
        "unit": "1M tokens",
        "base": 20.00,
        "addons": {"optimization": 25.00, "entity_resolution": 10.00, "deep_reasoning": 50.00},
    },
}

print("Latence Pricing Summary")
print("=" * 60)
for service, info in PRICING.items():
    base = info.get("base", info.get("default", "varies"))
    print(f"  {service:.<30} ${base} / {info['unit']}")
    if "addons" in info:
        for addon, cost in info["addons"].items():
            print(f"    + {addon:.<26} +${cost} / {info['unit']}")

## 1. Simulate a 10K-Document Pipeline

A realistic enterprise workload: process 10,000 documents (avg 5 pages, ~2000 tokens each) through DocInt → Entity Extraction → Relation Extraction.

In [ ]:
# TODO: Calculate cost for:
# - 10K docs * 5 pages avg = 50K pages through DocInt
# - 10K docs * 2000 tokens avg = 20M tokens through Entity Extraction
# - 10K docs * 2000 tokens avg = 20M tokens through Relation Extraction
# Show per-service and total cost, with and without addons

## 2. Compression ROI

Compression costs money. But it saves money on downstream LLM calls. When is it worth it?

Model: You compress documents before feeding to GPT-4 at ~$10/1M input tokens.

In [ ]:
# TODO: Calculate:
# - Cost of compressing 20M tokens at 50% rate
# - LLM cost without compression: 20M tokens * $10/1M
# - LLM cost with compression: 10M tokens * $10/1M
# - Net savings = LLM savings - compression cost
# - Break-even analysis: at what volume does compression pay for itself?

## 3. Feature Cost Comparison

Each service has optional addons that improve quality but add cost. When are they worth the premium?

In [ ]:
# TODO: For Entity Extraction, compare:
# - Base only ($1/1M tokens)
# - Base + auto labels ($6/1M tokens)
# - Base + auto labels + refinement ($16/1M tokens)
# Show cost multiplier and when each tier makes sense

## 4. Monthly Budget Projections

Project costs for small (1K docs/mo), medium (10K docs/mo), and large (100K docs/mo) workloads.

In [ ]:
# TODO: Build a projection table:
# | Workload  | DocInt | Entity Extraction | Relation Extraction | Embedding | Total/mo |
# |-----------|--------|-------------------|---------------------|-----------|----------|
# | 1K docs   | $...   | $...              | $...                | $...      | $...     |
# | 10K docs  | $...   | $...              | $...                | $...      | $...     |
# | 100K docs | $...   | $...              | $...                | $...      | $...     |

## 5. Live Cost Check

Run a small real request and verify the actual credit deduction matches the pricing model.

In [ ]:
# TODO: Check balance before, run a small embed request, check balance after
# Compare actual deduction with expected pricing

---
## Cleanup

In [ ]:
client.close()
print("Done.")